### Santa Barbara station request for refining to closest land gridcell
- SB station with current code process will pull a WRF gridcell that is over water
  - This produces a large discrepancy in values from observations that may impact bias-correction
- Test localizing SB station by pulling the closest **land** gridcell
  - Need to be considerate of elevation, mountains are very close to Santa Barbara
  - Need to be considerate of model spatial resolution, 9km will produce different result than 3km
- Produce:
  - Bias-corrected 3km netcdf file for Santa Barbara with both air temperature and dewpoint-temperature as adjusted variables
  - 1981 - 2100 period, SSP3-7.0, hourly, degF

In [ ]:
import climakitae as ck
import xarray as xr
import pandas as pd
import hvplot as hv
import numpy as np
import hvplot.pandas

from climakitae.util.utils import read_csv_file, get_closest_gridcell, convert_to_local_time, read_ae_colormap
import pkg_resources

from xclim.core.calendar import convert_calendar
from xclim.core.units import convert_units_to
from xclim.sdba.adjustment import QuantileDeltaMapping
from xclim.sdba import Grouper

from bokeh.models import HoverTool
from timezonefinder import TimezoneFinder

import s3fs

import pyproj
import itertools

# Running this nb with the tas+tdps localized data requires access to the private hadisd bucket

#### Step 1: Testing on `get_closest_gridcell` as is

In [ ]:
# grab station observations
station_id = '72392523190' # station id for KSBA

s3 = s3fs.S3FileSystem(anon=False, key=AWS_ACCESS_KEY_ID, secret=AWS_SECRET_ACCESS_KEY)
aws_path = "s3://wecc-hadisd/02_tmp_tas_dpt/"
filepath_zarr = aws_path + "HadISD_{}.zarr".format(station_id)

print('Opening: {}'.format(filepath_zarr))
store = s3fs.S3Map(root=filepath_zarr, s3=s3, check=False)
obs_ds = xr.open_zarr(store=store, consolidated=True)

obs_ds = obs_ds.tas
obs_ds = convert_units_to(obs_ds, "degF")
obs_ds = convert_calendar(obs_ds, "noleap")

# extract coordinates
lat0 = obs_ds.latitude.values
lon0 = obs_ds.longitude.values
print('Station location: ', lat0, lon0)

In [ ]:
# Initialize selections
selections = ck.Select()

# read in model output across 2 grids (9km, 3km)
selections.scenario_historical=['Historical Climate']
selections.scenario_ssp=['SSP 3-7.0 -- Business as Usual']
selections.append_historical = True
selections.area_average = 'No'
selections.time_slice = (1981, 2100)
selections.timescale = 'hourly'
selections.variable = 'Air Temperature at 2m'
selections.units = 'degF'
selections.cached_area = 'coordinate selection'
selections.latitude = (lat0-1., lat0+1.)
selections.longitude = (lon0-1., lon0+1.)
selections.resolution = '9 km'
wrf_ds_9km = selections.retrieve()

# localize to time zone
wrf_ds_9km = convert_to_local_time(wrf_ds_9km, selections)

# convert calendar
wrf_ds_9km = convert_calendar(wrf_ds_9km, "noleap")

# do some renaming for plotting ease later
wrf_ds_9km.attrs['physical_variable'] = wrf_ds_9km.name
wrf_ds_9km.name = '9 km WRF'

wrf_ds_9km = wrf_ds_9km.sel(time=slice('1981-01-01', '2100-12-31')) # ensuring right length

da_9km = get_closest_gridcell(wrf_ds_9km, lat0, lon0)

In [ ]:
# read in model output across 2 grids (9km, 3km)
selections.resolution = '3 km'

wrf_ds_3km = selections.retrieve()
# wrf_ds_3km = wrf_ds_9km.isel(simulation = 0).squeeze()

# localize to time zone
wrf_ds_3km = convert_to_local_time(wrf_ds_3km, selections)

# convert calendar
wrf_ds_3km = convert_calendar(wrf_ds_3km, "noleap")

# do some renaming for plotting ease later
wrf_ds_3km.attrs['physical_variable'] = wrf_ds_3km.name
wrf_ds_3km.name = '3 km WRF'

wrf_ds_3km = wrf_ds_3km.sel(time=slice('1981-01-01', '2100-12-31')) # ensuring right length

da_3km = get_closest_gridcell(wrf_ds_3km, lat0, lon0)

In [ ]:
wrf_ds = xr.merge([da_9km, da_3km], compat='override')
wrf_ds = wrf_ds.isel(simulation = 0).squeeze()

Note slightly different grid cell coordinates between 9km and 3km. 

In [ ]:
def group_ds(ds, obs_ds=obs_ds, projected_ceil=2100,
            metric='max'):
    
    proj_floor = str(projected_ceil-29)
    proj_ceil = str(projected_ceil)
    
    hist_ds = ds.sel(time=slice(str(obs_ds.time.values[0]),
            str(obs_ds.time.values[-1]))).groupby(
            'time.dayofyear')    
    ssp_ds = ds.sel(time=slice(proj_floor,proj_ceil)).groupby(
            'time.dayofyear')    
    obs_ds = obs_ds.groupby(
        'time.dayofyear')
    
    if (metric == 'max'):
        hist_ds = hist_ds.max()    
        ssp_ds = ssp_ds.max()    
        obs_ds = obs_ds.max()
    
    hist_ds = hist_ds.rename(dict(dayofyear = 'Day of Year'))
    ssp_ds = ssp_ds.rename(dict(dayofyear = 'Day of Year'))
    obs_ds = obs_ds.rename(dict(dayofyear = 'Day of Year'))
    
    return hist_ds, ssp_ds, obs_ds

def compare_raw_and_obs(ds, obs_ds=obs_ds, ylim=(None,None), 
                        width=700, height=300): 
    
    hist_gp, ssp_gp, obs_gp = group_ds(ds, obs_ds)
    
    hist_pl = hist_gp.hvplot(label='Historical raw',c='royalblue')    
    ssp_pl = ssp_gp.hvplot(label='Projected raw',c='goldenrod')    
    obs_pl = obs_gp.hvplot(label='Observations',c='k')
   
    pl = obs_pl * hist_pl * ssp_pl
    pl.opts(ylim=ylim, width=width, height=height, legend_position='right',
           toolbar='below', ylabel=obs_ds.units)
    
    return pl

In [ ]:
pl_9km = compare_raw_and_obs(wrf_ds['9 km WRF'], obs_ds=obs_ds).opts(title='WRF 9 km grid')
pl_3km = compare_raw_and_obs(wrf_ds['3 km WRF'], obs_ds=obs_ds).opts(title='WRF 3 km grid')

(pl_9km + pl_3km).cols(1).opts(title='KSBA daily max hourly 2m air temperature at 9 and 3 km WRF output, closest gridcell (water)')

Look at some maps to see wx station location + ID gridcells pulled

In [ ]:
da_9km_sub = wrf_ds_9km.isel(time = np.arange(0,366))
da_9km_land = da_9km_sub.where(da_9km_sub.landmask > 0., drop=False) # setting to False is critical so that landmask does not shift gridcells
da_9km_sub = ck.load(da_9km_sub)
da_9km_land = ck.load(da_9km_land)

da_3km_sub = wrf_ds_3km.isel(time = np.arange(0,366))
da_3km_land = da_3km_sub.where(da_3km_sub.landmask > 0., drop=False) # setting to False is critical so that landmask does not shift gridcells
da_3km_sub = ck.load(da_3km_sub)
da_3km_land = ck.load(da_3km_land)

In [ ]:
# add weather station as star to map
point_df = pd.DataFrame({
    "longitude (degrees_east)":[lon0],
    "latitude (degrees_north)":[lat0],
    "weather station": 'KSBA'
})

# add point for "selected gridcell as is"
grid_9df = pd.DataFrame({
    "longitude (degrees_east)":[-119.86], # hard coding in
    "latitude (degrees_north)":[34.37], # hard coding in
    "selected grid cell": '9km'
})

grid_3df = pd.DataFrame({
    "longitude (degrees_east)":[-119.85], # hard coding in
    "latitude (degrees_north)":[34.41], # hard coding in
    "selected grid cell": '3km'
}) 


# add point for "selected gridcell over land" -- these are very specific
grid_9LAND = pd.DataFrame({
    "longitude (degrees_east)":[-119.7775], # hard coding in
    "latitude (degrees_north)":[34.4214], # hard coding in
    "land grid cell": 'land_9km'
})

grid_3LAND = pd.DataFrame({
    "longitude (degrees_east)":[-119.8239], # hard coding in
    "latitude (degrees_north)":[34.4277], # hard coding in
    "land grid cell": 'land_3km'
}) 

In [ ]:
# 9km data view
p_all = ck.view(da_9km_sub) * grid_9df.hvplot.points(
    hover_cols = ['selected grid cell'], marker = 'circle', size = 50, color='blue') * point_df.hvplot.points(
    hover_cols = ['weather station'], marker = 'star', size = 200, color='black') * grid_9LAND.hvplot.points(
    hover_cols = ['selected grid cell'], marker = 'circle', size = 50, color = 'green')

p_land = ck.view(da_9km_land) * grid_9df.hvplot.points(
    hover_cols = ['selected grid cell'], marker = 'circle', size = 50, color='blue') * point_df.hvplot.points(
    hover_cols = ['weather station'], marker = 'star', size = 200, color='black') * grid_9LAND.hvplot.points(
    hover_cols = ['selected grid cell'], marker = 'circle', size = 50, color = 'green')

p_all + p_land

In [ ]:
# 3km data view
p_all = ck.view(da_3km_sub) * grid_3df.hvplot.points(
    hover_cols = ['selected grid cell'], marker = 'circle', size = 50, color='blue') * point_df.hvplot.points(
    hover_cols = ['weather station'], marker = 'star', size = 200, color='black') * grid_3LAND.hvplot.points(
    hover_cols = ['selected grid cell'], marker = 'circle', size = 50, color = 'green')


p_land = ck.view(da_3km_land) * grid_3df.hvplot.points(
    hover_cols = ['selected grid cell'], marker = 'circle', size = 50, color='blue') * point_df.hvplot.points(
    hover_cols = ['weather station'], marker = 'star', size = 200, color='black') * grid_3LAND.hvplot.points(
    hover_cols = ['selected grid cell'], marker = 'circle', size = 50, color = 'green')

p_all + p_land

---
Modifying get_closest_gridcell for SB handling

In [ ]:
def get_closest_land_gridcell(data, lat, lon, res='3km', print_coords=True):
    """From input gridded data, get the closest gridcell to a lat, lon coordinate pair.

    This function first transforms the lat,lon coords to the gridded data’s projection.
    Then, it uses xarray’s built in method .sel to get the nearest gridcell.

    Parameters
    -----------
    data: xr.DataArray or xr.Dataset
        Gridded data
    lat: float
        Latitude of coordinate pair
    lon: float
        Longitude of coordinate pair
    res: str
        Spatial resolution for Santa Barbara station
    print_coords: bool, optional
        Print closest coorindates?
        Default to True. Set to False for backend use.

    Returns
    --------
    xr.DataArray
        Grid cell closest to input lat,lon coordinate pair

    See also
    --------
    xarray.DataArray.sel
    """
    # Make Transformer object
    lat_lon_to_model_projection = pyproj.Transformer.from_crs(
        crs_from="epsg:4326",  # Lat/lon
        crs_to=data.rio.crs,  # Model projection
        always_xy=True,
    )

    # Hard-coding for Santa Barbara, forces land pixel selection over water pixel
    # Note fine-tuning produces different results for different spatial resolutions, specify which on call
    if lat==34.424 and lon==-119.842: # station coordinates for SB (HadISD_72392523190)
        print('Selecting closest land pixel for Santa Barbara...')
        if res == '9km':
            lat = 34.43
            lon = -119.83
        elif res == '3km':
            lat = 34.43
            lon = -119.845

    # Convert coordinates to x,y
    x, y = lat_lon_to_model_projection.transform(lon, lat)

    # Get closest gridcell
    closest_gridcell = data.sel(x=x, y=y, method="nearest")

    # Output information
    if print_coords:
        print(
            "Input coordinates: (%.4f, %.4f)" % (lat, lon)
            + "\nNearest grid cell coordinates: (%.4f, %.4f)"
            % (closest_gridcell.lat.values.item(), closest_gridcell.lon.values.item())
        )
    return closest_gridcell

In [ ]:
# Initialize selections
selections = ck.Select()

# testing with modified get_closest_land_gridcell
# read in model output across 2 grids (9km, 3km)
selections.scenario_historical=['Historical Climate']
selections.scenario_ssp=['SSP 3-7.0 -- Business as Usual']
selections.append_historical = True
selections.area_average = 'No'
selections.time_slice = (1981, 2100)
selections.timescale = 'hourly'
selections.variable = 'Air Temperature at 2m'
selections.units = 'degF'
selections.cached_area = 'coordinate selection'
selections.latitude = (lat0-1., lat0+1.)
selections.longitude = (lon0-1., lon0+1.)
selections.resolution = '9 km'

wrf_ds_9km = selections.retrieve()
# wrf_ds_9km = wrf_ds_9km.isel(simulation = 0).squeeze()

# localize to time zone
wrf_ds_9km = convert_to_local_time(wrf_ds_9km, selections)

# convert calendar
wrf_ds_9km = convert_calendar(wrf_ds_9km, "noleap")

# do some renaming for plotting ease later
wrf_ds_9km.attrs['physical_variable'] = wrf_ds_9km.name
wrf_ds_9km.name = '9 km WRF'

wrf_ds_9km = wrf_ds_9km.sel(time=slice('1981-01-01', '2100-12-31')) # ensuring right length

da_9km_SB = get_closest_land_gridcell(wrf_ds_9km, lat0, lon0, res='9km')

In [ ]:
# read in model output across 2 grids (9km, 3km)
selections.resolution = '3 km'

wrf_ds_3km = selections.retrieve()
# wrf_ds_3km = wrf_ds_9km.isel(simulation = 0).squeeze()

# localize to time zone
wrf_ds_3km = convert_to_local_time(wrf_ds_3km, selections)

# convert calendar
wrf_ds_3km = convert_calendar(wrf_ds_3km, "noleap")

# do some renaming for plotting ease later
wrf_ds_3km.attrs['physical_variable'] = wrf_ds_3km.name
wrf_ds_3km.name = '3 km WRF'

wrf_ds_3km = wrf_ds_3km.sel(time=slice('1981-01-01', '2100-12-31')) # ensuring right length

da_3km_SB = get_closest_land_gridcell(wrf_ds_3km, lat0, lon0, res='3km')

In [ ]:
wrf_ds_SB = xr.merge([da_9km_SB, da_3km_SB], compat='override')
wrf_ds_SB = wrf_ds_SB.isel(simulation = 0).squeeze()

In [ ]:
# now look at the difference between obs and land-9km / land-3km
pl_9km_SB = compare_raw_and_obs(wrf_ds_SB['9 km WRF'], obs_ds=obs_ds).opts(title='WRF 9 km land gridcell')
pl_3km_SB = compare_raw_and_obs(wrf_ds_SB['3 km WRF'], obs_ds=obs_ds).opts(title='WRF 3 km land gridcell')

(pl_9km_SB + pl_3km_SB).cols(1).opts(title='KSBA daily max hourly 2m air temperature at 9 and 3 km WRF output, land gridcell')

In [ ]:
# 3km does perform better than 9km